In [38]:
import numpy as np

from mpi4py import MPI
from petsc4py import PETSc

import dolfinx
import dolfinx.plot
import ufl
import basix.ufl

from fenicsx_ii import assemble_scalar, Circle, Average, assemble_matrix, LinearProblem, create_matrix, create_vector, assemble_vector

from mesh import GenMesh3D
from mesh3d1d import MeshData, Mesh3D1D

In [3]:
N = 4 
comm = MPI.COMM_WORLD

nodes, cells, graph_nodes, graph_cells, cells_edges, lm_index, boundary_indices = GenMesh3D(N)

marker_indices = np.insert(boundary_indices, 1, lm_index).astype(np.int32)

mesh_data = MeshData()

mesh_data.nodes = nodes
#print(nodes)
mesh_data.cells = cells

mesh_data.graph_nodes = graph_nodes
mesh_data.graph_cells = graph_cells
mesh_data.marker_indices = marker_indices
mesh_data.marker_values = np.array([2, 1, 2, 2], dtype=np.int32)

num_cells_in_edge = N//2
cells_array = np.arange(num_cells_in_edge, dtype=np.int32)
cell_indices = []
cell_indices.append(cells_array)
cell_indices.append(2*cells_array + num_cells_in_edge)
cell_indices.append(2*cells_array + 1 + num_cells_in_edge)

mesh_data.cell_indices = cell_indices

mesh_data.lm_index = lm_index
mesh_data.lm_indices = np.array([num_cells_in_edge, 0, 0], dtype=np.int32)

mesh_3d1d = Mesh3D1D(comm, mesh_data)

In [4]:
fs_3d = dolfinx.fem.functionspace(mesh_3d1d.mesh3D, ("DG", 1))
fs_edges = [dolfinx.fem.functionspace(edge_mesh, ("DG", 1)) for edge_mesh in mesh_3d1d.edge_meshes]
fs_lm = dolfinx.fem.functionspace(mesh_3d1d.lm_mesh, ("DG", 0))

function_spaces = ufl.MixedFunctionSpace(*[fs_3d, *fs_edges, fs_lm])

trial_funs = ufl.TrialFunctions(function_spaces)
u_3d = trial_funs[0]
u_1d = trial_funs[1:-1]
u_lm = trial_funs[-1]

test_funs = ufl.TestFunctions(function_spaces)
w_3d = test_funs[0]
w_1d = test_funs[1:-1]
w_lm = test_funs[-1]

In [5]:
x3d = ufl.SpatialCoordinate(mesh_3d1d.mesh3D)
x1d = ufl.SpatialCoordinate(mesh_3d1d.graph_mesh)

f_omega = 0

f_edges = [
    1,
    1,
    1,
]

In [ ]:
xi = 1
radius = 0.05
area = ufl.pi * radius ** 2
perimeter = 2 * ufl.pi * radius

q_degree = 20

sigma_omega = 20
sigma_e = 20
sigma_bifurc = 20

# initialize forms
a = [
    [ufl.ZeroBaseForm((ui, vj)) for vj in test_funs] for ui in trial_funs
]
L = [
    ufl.ZeroBaseForm((ui,)) for ui in test_funs
]

# 3D component

# measures
dx = ufl.Measure("dx", domain=mesh_3d1d.mesh3D)
dS = ufl.Measure("dS", domain=mesh_3d1d.mesh3D)
ds = ufl.Measure("ds", domain=mesh_3d1d.mesh3D)

h_omega = ufl.CellDiameter(mesh_3d1d.mesh3D)
h_omega_avg = (h_omega("+") + h_omega("-"))/2
n_omega = ufl.FacetNormal(mesh_3d1d.mesh3D)

# Bilinear Form
a[0][0] += ufl.inner(ufl.grad(u_3d), ufl.grad(w_3d)) * dx
a[0][0] -= ufl.inner(ufl.avg(ufl.grad(u_3d)), ufl.jump(w_3d, n_omega)) * dS
a[0][0] -= ufl.inner(ufl.avg(ufl.grad(w_3d)), ufl.jump(u_3d, n_omega)) * dS
a[0][0] += (sigma_omega / h_omega_avg) * ufl.inner(ufl.jump(u_3d, n_omega), ufl.jump(w_3d, n_omega)) * dS
a[0][0] -= ufl.inner(ufl.grad(u_3d), w_3d * n_omega) * ds
a[0][0] -= ufl.inner(ufl.grad(w_3d), u_3d * n_omega) * ds
a[0][0] += (sigma_omega / h_omega) * ufl.inner(u_3d, w_3d) * ds

# Linear Form
L[0] = ufl.inner(f_omega, w_3d) * dx

bifurc_marker = mesh_3d1d.bifurc_marker

# Edge components
    # Redefine forms to ensure they are properly categorized
for i, (edge, facet_marker) in enumerate(zip(mesh_3d1d.edge_meshes, mesh_3d1d.edge_facet_markers)):
    dx_edge = ufl.Measure("dx", domain=edge)
    dS_edge = ufl.Measure("dS", domain=edge)
    ds_edge = ufl.Measure("ds", domain=edge, subdomain_data=facet_marker)

    dx_bifurc = ufl.Measure("dx", mesh_3d1d.lm_mesh)

    # Use a constant scalar for h_e inside the 0D term to avoid compiling 1D variables on a 0D domain
    # We calculate the physical cell size (assuming uniform intervals in the submesh)
    coords = edge.geometry.x
    h_e_val = float(np.linalg.norm(coords[1] - coords[0]))

    h_e = ufl.CellDiameter(edge)
    h_e_avg = (h_e("+") + h_e("-"))/2
    n_e = ufl.FacetNormal(edge)

    # DG Form - Diagonal blocks
    a[i+1][i+1] += ufl.inner(ufl.grad(u_1d[i]), ufl.grad(w_1d[i])) * dx_edge
    a[i+1][i+1] -= ufl.inner(ufl.avg(ufl.grad(u_1d[i])), ufl.jump(w_1d[i], n_e)) * dS_edge
    a[i+1][i+1] -= ufl.inner(ufl.avg(ufl.grad(w_1d[i])), ufl.jump(u_1d[i], n_e)) * dS_edge
    a[i+1][i+1] += (sigma_e / h_e_avg) * ufl.inner(ufl.jump(u_1d[i], n_e), ufl.jump(w_1d[i], n_e)) * dS_edge
    #a[i+1][i+1] -= ufl.inner(ufl.grad(u_1d[i]), w_1d[i] * n_e) * ds_edge
    #a[i+1][i+1] -= ufl.inner(ufl.grad(w_1d[i]), u_1d[i] * n_e) * ds_edge
    #a[i+1][i+1] += (sigma_e / h_e) * ufl.inner(u_1d[i], w_1d[i]) * ds_edge

    # coupling terms
    restriction = Circle(edge, radius, q_degree)

    q_el = basix.ufl.quadrature_element(edge.basix_cell(), value_shape=(), degree=q_degree)
    restriction_space = dolfinx.fem.functionspace(edge, q_el)

    avg_u = Average(u_3d, restriction, restriction_space)
    avg_w = Average(w_3d, restriction, restriction_space)

    #a[i+1][i+1] += perimeter * xi * ufl.inner(avg_u, avg_w) * dx_edge
    a[0][i+1] -= perimeter * xi * ufl.inner(u_1d[i], avg_w) * dx_edge
    #a[i+1][i+1] -= perimeter * xi * ufl.inner(w_1d[i], avg_u) * dx_edge
    #a[i+1][i+1] += perimeter * xi * ufl.inner(u_1d[i], w_1d[i]) * dx_edge

    # Bifurcation node coupling
    a[i+1][i+1] += ufl.inner(ufl.grad(u_1d[i]), ufl.outer(w_1d[i], n_e)) * ds_edge(bifurc_marker)
    a[i+1][i+1] += ufl.inner(ufl.grad(w_1d[i]), ufl.outer(u_1d[i], n_e)) * ds_edge(bifurc_marker)
    a[i+1][i+1] += (sigma_bifurc / h_e) * ufl.inner(u_1d[i], w_1d[i]) * ds_edge(bifurc_marker)

    a[-1][i+1] -= ufl.inner(ufl.grad(u_1d[i]), ufl.outer(w_lm, n_e)) * ds_edge(bifurc_marker)
    a[i+1][-1] -= ufl.inner(ufl.grad(w_1d[i]), ufl.outer(u_lm, n_e)) * ds_edge(bifurc_marker)

    a[-1][i+1] -= (sigma_bifurc / h_e) * ufl.inner(u_1d[i], w_lm) * ds_edge(bifurc_marker)
    a[i+1][-1] -= (sigma_bifurc / h_e) * ufl.inner(w_1d[i], u_lm) * ds_edge(bifurc_marker)

    # Use h_e_val (the scalar float) here instead of the UFL object h_e
    a[-1][-1] += (sigma_bifurc / h_e_val) * u_lm * w_lm * dx_bifurc

    # RHS Form
    L[i+1] += ufl.inner(w_1d[i], f_edges[i]) * dx_edge
    #L[i+1] -= ufl.inner(ufl.grad(self._w_e[i]), ufl.outer(self._u_ex_edges[i], n_e)) * ds_edge(self._boundary_marker)
    #L[i+1] += (self._sigma_edge / h_e) * ufl.inner(self._u_ex_edges[i], self._w_e[i]) * ds_edge(self._boundary_marker)

entity_maps = mesh_3d1d.edge_entity_maps + [mesh_3d1d.lm_map] + mesh_3d1d.lm_to_edge_maps

for i, ai in enumerate(a):
    for j, aij in enumerate(ai):
          if isinstance(aij, ufl.ZeroBaseForm):
              a[i][j] = None

L_form = dolfinx.fem.form(L, entity_maps=entity_maps)
a_form = (a, entity_maps=entity_maps)



TypeError: Can only average arguments or coefficients, got <class 'ufl.referencevalue.ReferenceValue'>

In [ ]:
print(type())

<class 'ufl.argument.Argument'>


In [33]:
a_form = ufl.inner(ufl.grad(u_3d), ufl.grad(w_3d)) * dx
a_form -= ufl.inner(ufl.avg(ufl.grad(u_3d)), ufl.jump(w_3d, n_omega)) * dS
a_form -= ufl.inner(ufl.avg(ufl.grad(w_3d)), ufl.jump(u_3d, n_omega)) * dS
a_form += (sigma_omega / h_omega_avg) * ufl.inner(ufl.jump(u_3d, n_omega), ufl.jump(w_3d, n_omega)) * dS
a_form -= ufl.inner(ufl.grad(u_3d), w_3d * n_omega) * ds
a_form -= ufl.inner(ufl.grad(w_3d), u_3d * n_omega) * ds
a_form += (sigma_omega / h_omega) * ufl.inner(u_3d, w_3d) * ds

L_form = ufl.inner(f_omega, w_3d) * dx

for i, (edge, facet_marker) in enumerate(zip(mesh_3d1d.edge_meshes, mesh_3d1d.edge_facet_markers)):
    dx_edge = ufl.Measure("dx", domain=edge)
    dS_edge = ufl.Measure("dS", domain=edge)
    ds_edge = ufl.Measure("ds", domain=edge, subdomain_data=facet_marker)

    dx_bifurc = ufl.Measure("dx", mesh_3d1d.lm_mesh)

    # Use a constant scalar for h_e inside the 0D term to avoid compiling 1D variables on a 0D domain
    # We calculate the physical cell size (assuming uniform intervals in the submesh)
    coords = edge.geometry.x
    h_e_val = float(np.linalg.norm(coords[1] - coords[0]))

    h_e = ufl.CellDiameter(edge)
    h_e_avg = (h_e("+") + h_e("-"))/2
    n_e = ufl.FacetNormal(edge)

    # DG Form - Diagonal blocks
    a_form += ufl.inner(ufl.grad(u_1d[i]), ufl.grad(w_1d[i])) * dx_edge
    a_form -= ufl.inner(ufl.avg(ufl.grad(u_1d[i])), ufl.jump(w_1d[i], n_e)) * dS_edge
    a_form -= ufl.inner(ufl.avg(ufl.grad(w_1d[i])), ufl.jump(u_1d[i], n_e)) * dS_edge
    a_form += (sigma_e / h_e_avg) * ufl.inner(ufl.jump(u_1d[i], n_e), ufl.jump(w_1d[i], n_e)) * dS_edge

    # coupling terms
    restriction = Circle(edge, radius, q_degree)

    q_el = basix.ufl.quadrature_element(edge.basix_cell(), value_shape=(), degree=q_degree)
    restriction_space = dolfinx.fem.functionspace(edge, q_el)

    avg_u = Average(u_3d, restriction, restriction_space)
    avg_w = Average(w_3d, restriction, restriction_space)

    a_form += perimeter * xi * ufl.inner(avg_u, avg_w) * dx_edge
    a_form -= perimeter * xi * ufl.inner(u_1d[i], avg_w) * dx_edge
    a_form -= perimeter * xi * ufl.inner(w_1d[i], avg_u) * dx_edge
    a_form += perimeter * xi * ufl.inner(u_1d[i], w_1d[i]) * dx_edge

    # Bifurcation node coupling
    a_form += ufl.inner(ufl.grad(u_1d[i]), ufl.outer(w_1d[i], n_e)) * ds_edge(bifurc_marker)
    a_form += ufl.inner(ufl.grad(w_1d[i]), ufl.outer(u_1d[i], n_e)) * ds_edge(bifurc_marker)
    a_form += (sigma_bifurc / h_e) * ufl.inner(u_1d[i], w_1d[i]) * ds_edge(bifurc_marker)

    a_form -= ufl.inner(ufl.grad(u_1d[i]), ufl.outer(w_lm, n_e)) * ds_edge(bifurc_marker)
    a_form -= ufl.inner(ufl.grad(w_1d[i]), ufl.outer(u_lm, n_e)) * ds_edge(bifurc_marker)

    a_form -= (sigma_bifurc / h_e) * ufl.inner(u_1d[i], w_lm) * ds_edge(bifurc_marker)
    a_form -= (sigma_bifurc / h_e) * ufl.inner(w_1d[i], u_lm) * ds_edge(bifurc_marker)

    # Use h_e_val (the scalar float) here instead of the UFL object h_e
    a_form += (sigma_bifurc / h_e_val) * u_lm * w_lm * dx_bifurc

    L_form += ufl.inner(w_1d[i], f_edges[i]) * dx_edge


In [ ]:

A_mat = create_matrix(a_form)
A_mat.zeroEntries()
assemble_matrix(A_mat, a_form, bcs=[])
A_mat.assemble()

# Extract function spaces and create the block RHS vector
spaces = dolfinx.fem.extract_function_spaces(L_form)
kind = "nest" if A_mat.getType() == "nest" else None

b_vec = create_vector(spaces, kind=kind)
with b_vec.localForm() as loc_b:
    loc_b.set(0)
assemble_vector(b_vec, L_form)
b_vec.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)

# Create block solution vector
x = create_vector(spaces, kind=kind)

# Configure and run the PETSc solver directly
ksp = PETSc.KSP().create(comm=mesh_3d1d.comm)
ksp.setOperators(A_mat)
ksp.setType("preonly")
pc = ksp.getPC()
pc.setType("lu")
pc.setFactorSolverType("mumps")

ksp.solve(b_vec, x)

functions = []

for i, edge in enumerate(fs_edges):
    functions.append(dolfinx.fem.Function(edge, name=f"edge_{i}"))
functions.append(dolfinx.fem.Function(fs_lm, name="lm_value"))

dolfinx.la.petsc._ghost_update(
    x,
    insert_mode=PETSc.InsertMode.INSERT,
    scatter_mode=PETSc.ScatterMode.FORWARD,
)

dolfinx.fem.petsc.assign(x, functions)

AttributeError: 'Form' object has no attribute '_cpp_object'

In [36]:
petsc_options = {
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
    "ksp_error_if_not_converged": True,
}
problem = LinearProblem(
    a_form,
    L_form,
    petsc_options_prefix="coupled_poisson",
    petsc_options=petsc_options,
)
funs = problem.solve()

AssertionError: 